# Wave 01 - step 2: transform in Snowflake

Calls the Snowflake stored procedure that transforms the staging table into the final tables. Runs as task `transform` of every wave_01 job, after `01_extract_load`. Parameters come from `resources/jobs/wave_01_*.job.yml`.

In [0]:
%pip install -q snowflake-connector-python
dbutils.library.restartPython()

In [0]:
import os

# Defaults are for interactive runs; job parameters override them.
_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
_default_src = "/Workspace" + _nb_path.split("/notebooks/")[0] + "/src"

dbutils.widgets.text("job_name", "wave_01_interactive")
dbutils.widgets.text("src_path", _default_src)
dbutils.widgets.text("catalog", "dev_catalog")
dbutils.widgets.text("schema", "talend_migration")
dbutils.widgets.text("snowflake_secret_scope", "snowflake-dev")
dbutils.widgets.text("snowflake_schema", "")  # required: your dev shadow schema, e.g. MIGRATION_DEV_<short_name>_main
dbutils.widgets.text("snowflake_procedure", "")

p = {k: dbutils.widgets.get(k) for k in [
    "job_name", "src_path", "catalog", "schema", "snowflake_secret_scope", "snowflake_schema", "snowflake_procedure",
]}
print(p)

In [0]:
import sys

if p["src_path"] not in sys.path:
    sys.path.insert(0, p["src_path"])

from operators import SnowflakeConfig, call_procedure

if not p["snowflake_schema"]:
    raise ValueError("Set the snowflake_schema widget to your dev shadow schema (e.g. MIGRATION_DEV_<short_name>_main)")
# The session schema is the shadow schema; unqualified procedure and table names resolve here.
sf_cfg = SnowflakeConfig.from_secrets(dbutils, scope=p["snowflake_secret_scope"], schema=p["snowflake_schema"])

In [0]:
result = call_procedure(sf_cfg, p["snowflake_procedure"], [p["job_name"]])
print(f"{p['snowflake_procedure']} returned: {result}")

In [0]:
import json

dbutils.notebook.exit(json.dumps({"job_name": p["job_name"], "procedure_result": str(result)}))